In [ ]:
import os
import random
import time
from datetime import UTC, datetime

import pandas as pd
import psycopg2
from dotenv import load_dotenv
from faker import Faker


In [ ]:
def run_rules(df):
    """Apply simple fraud rules to one generated transaction."""
    df = df.copy()

    # Start from the normal path and override it only when a rule is triggered.
    df["rules_triggered"] = "No Rules Triggered"
    df["rules_explanation"] = None
    df["decision"] = "Approved"

    amount = float(df.loc[0, "amount"])
    is_blacklisted = bool(df.loc[0, "account_blacklisted"])
    is_real_time = df.loc[0, "trans_type"] == "Real_time_transaction"

    # Blacklisted accounts take priority over the amount threshold.
    if is_real_time and is_blacklisted:
        df.loc[0, "rules_triggered"] = "Rule2"
        df.loc[0, "rules_explanation"] = "The account is blacklisted"
        df.loc[0, "decision"] = "Rejected"
    elif is_real_time and amount >= 100:
        df.loc[0, "rules_triggered"] = "Rule1"
        df.loc[0, "rules_explanation"] = "Real-time transaction amount is at least 100"
        df.loc[0, "decision"] = "Rejected"

    return df.iloc[0].to_dict()


In [ ]:
fake = Faker()

# Keep the batch small so Grafana updates regularly without overwhelming the database.
BATCH_SIZE = 10
SLEEP_SECONDS = 15

merchant_categories = [
    "Retail",
    "Electronics",
    "Clothing",
    "Groceries",
    "Pharmacy",
    "Entertainment",
    "Dining",
    "Travel",
    "Utilities",
    "Healthcare",
]
card_types = {
    "visa": "visa",
    "mastercard": "mastercard",
}

# Load credentials from .env locally; config.sample documents the required keys.
load_dotenv()
db_config = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": int(os.getenv("DB_PORT", 5432)),
    "dbname": os.getenv("DB_NAME", "postgres"),
    "user": os.getenv("DB_USER", "postgres"),
    "password": os.getenv("DB_PASSWORD", "postgres"),
}

conn = psycopg2.connect(**db_config)
cur = conn.cursor()

# Keep the raw transaction fields and rule outputs in the same table for simple Grafana queries.
create_table_query = """
CREATE TABLE IF NOT EXISTS banking_data (
    id SERIAL PRIMARY KEY,
    timestamp TIMESTAMPTZ NOT NULL,
    uniq_id UUID NOT NULL,
    trans_type VARCHAR(50) NOT NULL,
    amount DECIMAL(10, 2) NOT NULL,
    amount_crr DECIMAL(10, 2) NOT NULL,
    account_holder_name VARCHAR(100) NOT NULL,
    card_presence VARCHAR(50) NOT NULL,
    merchant_category VARCHAR(50) NOT NULL,
    card_type VARCHAR(50) NOT NULL,
    card_id VARCHAR(20) NOT NULL,
    account_id UUID NOT NULL,
    account_blacklisted BOOLEAN NOT NULL,
    rules_triggered VARCHAR(100),
    rules_explanation VARCHAR(100),
    decision VARCHAR(100)
);
"""
cur.execute(create_table_query)

# Rename the old misspelled column if this notebook created the table earlier.
cur.execute("""
DO $$
BEGIN
    IF EXISTS (
        SELECT 1
        FROM information_schema.columns
        WHERE table_schema = current_schema()
          AND table_name = 'banking_data'
          AND column_name = 'card_presense'
    ) AND NOT EXISTS (
        SELECT 1
        FROM information_schema.columns
        WHERE table_schema = current_schema()
          AND table_name = 'banking_data'
          AND column_name = 'card_presence'
    ) THEN
        ALTER TABLE banking_data RENAME COLUMN card_presense TO card_presence;
    END IF;
END $$;
""")
conn.commit()


def generate_record():
    # Faker keeps the demo realistic without using any real customer data.
    card_type = random.choice(list(card_types.keys()))
    return {
        "uniq_id": [fake.uuid4()],
        "trans_type": [random.choice(["Real_time_transaction", "settlements", "dispute"])],
        "amount": [round(random.uniform(10.0, 1000.0), 2)],
        "amount_crr": [round(random.uniform(10.0, 1000.0), 2)],
        "account_holder_name": [fake.name()],
        "card_presence": [random.choice(["Present", "Not Present"])],
        "merchant_category": [random.choice(merchant_categories)],
        "card_type": [card_type],
        "card_id": [fake.credit_card_number(card_type=card_types[card_type])],
        "account_id": [fake.uuid4()],
        "account_blacklisted": [random.choice([True, False])],
    }


# Run continuously so Grafana has a small stream of fresh records to visualize.
try:
    while True:
        timestamp = datetime.now(UTC)

        for _ in range(BATCH_SIZE):
            df = pd.DataFrame(generate_record())
            record = run_rules(df)
            # Parameterized SQL keeps values separate from the query text.
            cur.execute(
                """
                INSERT INTO banking_data (
                    timestamp, uniq_id, trans_type, amount, amount_crr,
                    account_holder_name, card_presence, merchant_category,
                    card_type, card_id, account_id, account_blacklisted,
                    rules_triggered, rules_explanation, decision
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    timestamp,
                    record["uniq_id"],
                    record["trans_type"],
                    record["amount"],
                    record["amount_crr"],
                    record["account_holder_name"],
                    record["card_presence"],
                    record["merchant_category"],
                    record["card_type"],
                    record["card_id"],
                    record["account_id"],
                    record["account_blacklisted"],
                    record["rules_triggered"],
                    record["rules_explanation"],
                    record["decision"],
                ),
            )

        conn.commit()
        print(f"Inserted {BATCH_SIZE} records at {timestamp.isoformat()}")
        time.sleep(SLEEP_SECONDS)
except KeyboardInterrupt:
    print("Data generation stopped by user.")
finally:
    # Close database resources even when the notebook cell is interrupted.
    cur.close()
    conn.close()
